In [2]:
TRAIN_PATH = "cvefixes_10k_train.csv"

In [5]:
import pandas as pd

TRAIN_PATH = "cvefixes_10k_train.csv"

df = pd.read_csv(TRAIN_PATH)
print("Shape:", df.shape)
print("Columns:")
for i, c in enumerate(df.columns):
    print(i, c)

Shape: (8000, 29)
Columns:
0 cve_id
1 hash
2 repo_url
3 cve_description
4 cvss2_base_score
5 cvss3_base_score
6 published_date
7 severity
8 cwe_id
9 cwe_name
10 cwe_description
11 commit_message
12 commit_date
13 version_tag
14 repo_total_files
15 repo_total_commits
16 file_paths
17 language
18 diff_stats
19 diff_with_context
20 vulnerable_code
21 fixed_code
22 security_keywords
23 sample_id
24 verdict
25 confidence
26 rag_score
27 rag_signals
28 evidence_snippet


In [8]:
import os
import pandas as pd
import numpy as np

TRAIN_PATH = "cvefixes_10k_train.csv"
TEST_PATH  = "cvefixes_10k_test.csv"

OUT_TRAIN = "cvefixes_10k_train_code_label.csv"
OUT_TEST  = "cvefixes_10k_test_code_label.csv"

def make_code_label(df: pd.DataFrame) -> pd.DataFrame:
    req = ["vulnerable_code", "fixed_code"]
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    base_cols = [c for c in df.columns if c not in ["vulnerable_code", "fixed_code", "label", "code"]]

    pos = df[base_cols].copy()
    pos["code"] = df["vulnerable_code"].astype(str)
    pos["label"] = 1

    neg = df[base_cols].copy()
    neg["code"] = df["fixed_code"].astype(str)
    neg["label"] = 0

    out = pd.concat([pos, neg], ignore_index=True)

    # keep ordering: for each original row, vulnerable then fixed
    # (concat pos then neg keeps all positives first; so we interleave)
    out = pd.concat(
        [pos.assign(_ord=np.arange(len(pos))*2), neg.assign(_ord=np.arange(len(neg))*2 + 1)],
        ignore_index=True
    ).sort_values("_ord", kind="stable").drop(columns=["_ord"]).reset_index(drop=True)

    # drop empty code rows if any
    out["code"] = out["code"].astype(str)
    out = out[out["code"].str.strip().ne("")].reset_index(drop=True)
    return out

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

train_out = make_code_label(train_df)
test_out  = make_code_label(test_df)

train_out.to_csv(OUT_TRAIN, index=False)
test_out.to_csv(OUT_TEST, index=False)

print("Saved:", OUT_TRAIN, "rows:", len(train_out))
print("Saved:", OUT_TEST,  "rows:", len(test_out))
print("\nTrain label counts:\n", train_out["label"].value_counts())
print("\nTest label counts:\n", test_out["label"].value_counts())
print("\nColumns:", list(train_out.columns))

Saved: cvefixes_10k_train_code_label.csv rows: 15986
Saved: cvefixes_10k_test_code_label.csv rows: 3997

Train label counts:
 label
0    8000
1    7986
Name: count, dtype: int64

Test label counts:
 label
0    2000
1    1997
Name: count, dtype: int64

Columns: ['cve_id', 'hash', 'repo_url', 'cve_description', 'cvss2_base_score', 'cvss3_base_score', 'published_date', 'severity', 'cwe_id', 'cwe_name', 'cwe_description', 'commit_message', 'commit_date', 'version_tag', 'repo_total_files', 'repo_total_commits', 'file_paths', 'language', 'diff_stats', 'diff_with_context', 'security_keywords', 'sample_id', 'verdict', 'confidence', 'rag_score', 'rag_signals', 'evidence_snippet', 'code', 'label']


In [13]:
import pandas as pd
import numpy as np

SEED = 42

TRAIN_PATH = "cvefixes_10k_train_code_label.csv"
TEST_PATH  = "cvefixes_10k_test_code_label.csv"

OUT_TRAIN = "cvefixes_train_2k_balanced.csv"   # 1000 label=1 + 1000 label=0
OUT_TEST  = "cvefixes_test_400_balanced.csv"   # 200 label=1 + 200 label=0

LABEL_COL = "label"   # change if needed


def balanced_sample(df: pd.DataFrame, n_pos: int, n_neg: int, seed: int = 42) -> pd.DataFrame:
    if LABEL_COL not in df.columns:
        raise ValueError(
            f"Column '{LABEL_COL}' not found. Available columns: {list(df.columns)}"
        )

    df = df.copy()

    # Convert label column safely to numeric
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")

    # Remove NaN / inf labels
    df = df[df[LABEL_COL].notna()]
    df = df[np.isfinite(df[LABEL_COL])]

    # Keep only binary labels 0 and 1
    df = df[df[LABEL_COL].isin([0, 1])]

    # Convert to int after cleaning
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    pos = df[df[LABEL_COL] == 1]
    neg = df[df[LABEL_COL] == 0]

    if len(pos) < n_pos:
        raise ValueError(f"Not enough positives: need {n_pos}, have {len(pos)}")
    if len(neg) < n_neg:
        raise ValueError(f"Not enough negatives: need {n_neg}, have {len(neg)}")

    pos_s = pos.sample(n=n_pos, random_state=seed)
    neg_s = neg.sample(n=n_neg, random_state=seed)

    out = pd.concat([pos_s, neg_s], ignore_index=True)
    out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    return out


train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train label raw unique values:")
print(train_df[LABEL_COL].value_counts(dropna=False).head(20))

print("\nTest label raw unique values:")
print(test_df[LABEL_COL].value_counts(dropna=False).head(20))

train_bal = balanced_sample(train_df, n_pos=1000, n_neg=1000, seed=SEED)
test_bal  = balanced_sample(test_df, n_pos=200, n_neg=200, seed=SEED)

train_bal.to_csv(OUT_TRAIN, index=False)
test_bal.to_csv(OUT_TEST, index=False)

print("\nSaved:", OUT_TRAIN, "shape:", train_bal.shape, "labels:", train_bal[LABEL_COL].value_counts().to_dict())
print("Saved:", OUT_TEST, "shape:", test_bal.shape, "labels:", test_bal[LABEL_COL].value_counts().to_dict())

Train label raw unique values:
label
0.0    7996
1.0    7973
NaN      34
Name: count, dtype: int64

Test label raw unique values:
label
0.0    1999
1.0    1996
NaN       4
Name: count, dtype: int64

Saved: cvefixes_train_2k_balanced.csv shape: (2000, 29) labels: {0: 1000, 1: 1000}
Saved: cvefixes_test_400_balanced.csv shape: (400, 29) labels: {0: 200, 1: 200}


In [15]:
import os, random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix
)

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model

# ----------------------------
# PATHS (your drive)
# ----------------------------

TRAIN_PATH = "cvefixes_train_2k_balanced.csv"   # 1000 label=1 + 1000 label=0
TEST_PATH  = "cvefixes_test_400_balanced.csv"


# where to save adapter
OUT_DIR = "transaction_latest/deepseekcoder13b_CVEvulV_qlora"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# CONFIG
# ----------------------------
TEXT_COL  = "code"
LABEL_COL = "label"

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

SEED = 42
MAX_LEN = 512         # 1024 করলে slow/oom হতে পারে
EPOCHS = 5            # 5 দিলে অনেক সময় লাগবে
LR = 2e-4
TRAIN_BSZ = 2
GRAD_ACC = 8          # effective batch ~16

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ----------------------------
# Load data
# ----------------------------
train_df = pd.read_csv(TRAIN_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()
test_df  = pd.read_csv(TEST_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()
train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)
test_df[LABEL_COL]  = test_df[LABEL_COL].astype(int)

print("Train:", train_df.shape, train_df[LABEL_COL].value_counts().to_dict())
print("Test :", test_df.shape,  test_df[LABEL_COL].value_counts().to_dict())

# ----------------------------
# Prompt (base-friendly)
# ----------------------------
def make_prompt(code: str) -> str:
    return (
        "Task: Vulnerability classification.\n"
        "Return a single digit only: 0 (non-vulnerable) or 1 (vulnerable).\n\n"
        "Code:\n"
        f"{code}\n\n"
        "Label:"
    )

# ----------------------------
# Tokenizer
# ----------------------------
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# single-token label IDs
id_0 = tok.encode("0", add_special_tokens=False)
id_1 = tok.encode("1", add_special_tokens=False)

if len(id_0) != 1 or len(id_1) != 1:
    # fallback to " yes"/" no" (usually single tokens)
    id_no  = tok.encode(" no", add_special_tokens=False)
    id_yes = tok.encode(" yes", add_special_tokens=False)
    if len(id_no) != 1 or len(id_yes) != 1:
        raise RuntimeError("Neither '0/1' nor ' yes/no' are single-token labels for this tokenizer.")
    print("[WARN] Using ' yes/no' labels instead of 0/1")
    label_token_map = {0: id_no[0], 1: id_yes[0]}
    def make_prompt(code: str) -> str:
        return (
            "Task: Vulnerability classification.\n"
            "Return ONLY one token: ' yes' (vulnerable) or ' no' (non-vulnerable).\n\n"
            "Code:\n"
            f"{code}\n\n"
            "Label:"
        )
else:
    label_token_map = {0: id_0[0], 1: id_1[0]}

# ----------------------------
# Dataset (supervise only last token)
# ----------------------------
class VulClsCausalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.codes = frame[TEXT_COL].astype(str).tolist()
        self.ys = frame[LABEL_COL].astype(int).tolist()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        code = self.codes[idx]
        y = int(self.ys[idx])

        prompt = make_prompt(code)
        prompt_ids = tok(
            prompt,
            truncation=True,
            max_length=MAX_LEN - 1,
            add_special_tokens=True
        )["input_ids"]

        label_id = int(label_token_map[y])
        input_ids = prompt_ids + [label_id]
        labels = [-100] * len(prompt_ids) + [label_id]
        attn = [1] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attn, labels = [], [], []
    for x in batch:
        pad = max_len - len(x["input_ids"])
        input_ids.append(torch.cat([x["input_ids"], torch.full((pad,), tok.pad_token_id, dtype=torch.long)]))
        attn.append(torch.cat([x["attention_mask"], torch.zeros((pad,), dtype=torch.long)]))
        labels.append(torch.cat([x["labels"], torch.full((pad,), -100, dtype=torch.long)]))
    return {"input_ids": torch.stack(input_ids), "attention_mask": torch.stack(attn), "labels": torch.stack(labels)}

train_ds = VulClsCausalDataset(train_df)

# ----------------------------
# Load 4-bit model (QLoRA)
# ----------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# stability/memory
model.config.use_cache = False
try:
    model.gradient_checkpointing_enable()
except Exception:
    pass

# ----------------------------
# LoRA
# ----------------------------
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ----------------------------
# Train
# ----------------------------
args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=TRAIN_BSZ,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=50,
    save_steps=300,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collate_fn,
)

trainer.train()

# ----------------------------
# Test inference + metrics
# ----------------------------
@torch.no_grad()
def predict_probs_and_labels(frame: pd.DataFrame, batch_size=8):
    model.eval()
    probs_1, preds = [], []
    y_true = frame[LABEL_COL].astype(int).to_numpy()

    for start in range(0, len(frame), batch_size):
        batch_codes = frame[TEXT_COL].iloc[start:start+batch_size].astype(str).tolist()
        prompts = [make_prompt(c) for c in batch_codes]

        enc = tok(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        ).to(model.device)

        out = model(**enc)

        last_idx = enc["attention_mask"].sum(dim=1) - 1
        logits_last = out.logits[torch.arange(out.logits.size(0), device=out.logits.device), last_idx, :]

        log0 = logits_last[:, label_token_map[0]]
        log1 = logits_last[:, label_token_map[1]]
        p1 = torch.softmax(torch.stack([log0, log1], dim=1), dim=1)[:, 1]

        probs_1.extend(p1.detach().float().cpu().tolist())
        preds.extend((p1 >= 0.5).long().cpu().tolist())

    return np.array(probs_1), np.array(preds), y_true

probs_1, y_pred, y_true = predict_probs_and_labels(test_df, batch_size=8)

acc  = accuracy_score(y_true, y_pred)
pre  = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)
mcc  = matthews_corrcoef(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
spe = tn / (tn + fp + 1e-12)

auc = roc_auc_score(y_true, probs_1) if len(np.unique(y_true)) == 2 else float("nan")

# print like your table header
print("\nAcc.\tPre.\tRec.\tF1.\tSpe.\tMCC\tAUC")
print(f"{acc:.4f}\t{pre:.4f}\t{rec:.4f}\t{f1:.4f}\t{spe:.4f}\t{mcc:.4f}\t{auc:.4f}")

print("Confusion [tn fp fn tp]:", tn, fp, fn, tp)

# ----------------------------
# Save adapter + tokenizer
# ----------------------------
adapter_dir = os.path.join(OUT_DIR, "lora_adapter")
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)

print("Saved LoRA adapter + tokenizer to:", adapter_dir)


Train: (1692, 2) {0: 922, 1: 770}
Test : (343, 2) {0: 190, 1: 153}


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 14,991,360 || all params: 1,361,463,296 || trainable%: 1.1011


Step,Training Loss
50,3.421037
100,0.699841
150,0.658734
200,0.634405
250,0.559317
300,0.517211
350,0.420275
400,0.358900
450,0.233667
500,0.204762



Acc.	Pre.	Rec.	F1.	Spe.	MCC	AUC
0.7085	0.6646	0.6993	0.6815	0.7158	0.4135	0.7757
Confusion [tn fp fn tp]: 136 54 46 107
Saved LoRA adapter + tokenizer to: transaction_latest/deepseekcoder13b_CVEvulV_qlora/lora_adapter


In [1]:
import os, random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix
)

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ----------------------------
# PATHS
# ----------------------------
TRAIN_PATH = "cvefixes_train_2k_balanced.csv"
TEST_PATH  = "cvefixes_test_400_balanced.csv"

OUT_DIR = "transaction_latest/qwen25coder15b_CVEvul_qlora"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# CONFIG
# ----------------------------
TEXT_COL  = "code"
LABEL_COL = "label"

# Official Qwen base coder model
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B"

SEED = 42
MAX_LEN = 512
EPOCHS = 5
LR = 2e-4
TRAIN_BSZ = 2
GRAD_ACC = 8

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ----------------------------
# Load data
# ----------------------------
train_df = pd.read_csv(TRAIN_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()
test_df  = pd.read_csv(TEST_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()

train_df[LABEL_COL] = pd.to_numeric(train_df[LABEL_COL], errors="coerce")
test_df[LABEL_COL]  = pd.to_numeric(test_df[LABEL_COL], errors="coerce")

train_df = train_df[train_df[LABEL_COL].isin([0, 1])].copy()
test_df  = test_df[test_df[LABEL_COL].isin([0, 1])].copy()

train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)
test_df[LABEL_COL]  = test_df[LABEL_COL].astype(int)

print("Train:", train_df.shape, train_df[LABEL_COL].value_counts().to_dict())
print("Test :", test_df.shape,  test_df[LABEL_COL].value_counts().to_dict())

# ----------------------------
# Prompt
# ----------------------------
def make_prompt(code: str) -> str:
    return (
        "Task: Vulnerability classification.\n"
        "Return a single digit only: 0 (non-vulnerable) or 1 (vulnerable).\n\n"
        "Code:\n"
        f"{code}\n\n"
        "Label:"
    )

# ----------------------------
# Tokenizer
# ----------------------------
tok = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=True
)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

tok.padding_side = "right"

# ----------------------------
# Label token IDs
# ----------------------------
id_0 = tok.encode("0", add_special_tokens=False)
id_1 = tok.encode("1", add_special_tokens=False)

if len(id_0) != 1 or len(id_1) != 1:
    id_no  = tok.encode(" no", add_special_tokens=False)
    id_yes = tok.encode(" yes", add_special_tokens=False)

    if len(id_no) != 1 or len(id_yes) != 1:
        raise RuntimeError("Neither '0/1' nor ' yes/no' are single-token labels for this tokenizer.")

    print("[WARN] Using ' yes/no' labels instead of 0/1")
    label_token_map = {0: id_no[0], 1: id_yes[0]}

    def make_prompt(code: str) -> str:
        return (
            "Task: Vulnerability classification.\n"
            "Return ONLY one token: ' yes' (vulnerable) or ' no' (non-vulnerable).\n\n"
            "Code:\n"
            f"{code}\n\n"
            "Label:"
        )
else:
    label_token_map = {0: id_0[0], 1: id_1[0]}

# ----------------------------
# Dataset
# ----------------------------
class VulClsCausalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.codes = frame[TEXT_COL].astype(str).tolist()
        self.ys = frame[LABEL_COL].astype(int).tolist()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        code = self.codes[idx]
        y = int(self.ys[idx])

        prompt = make_prompt(code)
        prompt_ids = tok(
            prompt,
            truncation=True,
            max_length=MAX_LEN - 1,
            add_special_tokens=True
        )["input_ids"]

        label_id = int(label_token_map[y])
        input_ids = prompt_ids + [label_id]
        labels = [-100] * len(prompt_ids) + [label_id]
        attn = [1] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attn, labels = [], [], []

    for x in batch:
        pad = max_len - len(x["input_ids"])
        input_ids.append(
            torch.cat([x["input_ids"], torch.full((pad,), tok.pad_token_id, dtype=torch.long)])
        )
        attn.append(
            torch.cat([x["attention_mask"], torch.zeros((pad,), dtype=torch.long)])
        )
        labels.append(
            torch.cat([x["labels"], torch.full((pad,), -100, dtype=torch.long)])
        )

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attn),
        "labels": torch.stack(labels),
    }

train_ds = VulClsCausalDataset(train_df)

# ----------------------------
# Load 4-bit model (QLoRA)
# ----------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

# stability/memory
model.config.use_cache = False
try:
    model.gradient_checkpointing_enable()
except Exception:
    pass

# ----------------------------
# LoRA
# ----------------------------
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ----------------------------
# Train
# ----------------------------
args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=TRAIN_BSZ,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=50,
    save_steps=300,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collate_fn,
)

trainer.train()

# ----------------------------
# Test inference + metrics
# ----------------------------
@torch.no_grad()
def predict_probs_and_labels(frame: pd.DataFrame, batch_size=8):
    model.eval()
    probs_1, preds = [], []
    y_true = frame[LABEL_COL].astype(int).to_numpy()

    for start in range(0, len(frame), batch_size):
        batch_codes = frame[TEXT_COL].iloc[start:start+batch_size].astype(str).tolist()
        prompts = [make_prompt(c) for c in batch_codes]

        enc = tok(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        ).to(model.device)

        out = model(**enc)

        last_idx = enc["attention_mask"].sum(dim=1) - 1
        logits_last = out.logits[
            torch.arange(out.logits.size(0), device=out.logits.device),
            last_idx,
            :
        ]

        log0 = logits_last[:, label_token_map[0]]
        log1 = logits_last[:, label_token_map[1]]
        p1 = torch.softmax(torch.stack([log0, log1], dim=1), dim=1)[:, 1]

        probs_1.extend(p1.detach().float().cpu().tolist())
        preds.extend((p1 >= 0.5).long().cpu().tolist())

    return np.array(probs_1), np.array(preds), y_true

probs_1, y_pred, y_true = predict_probs_and_labels(test_df, batch_size=8)

acc  = accuracy_score(y_true, y_pred)
pre  = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)
mcc  = matthews_corrcoef(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
spe = tn / (tn + fp + 1e-12)

auc = roc_auc_score(y_true, probs_1) if len(np.unique(y_true)) == 2 else float("nan")

print("\nAcc.\tPre.\tRec.\tF1.\tSpe.\tMCC\tAUC")
print(f"{acc:.4f}\t{pre:.4f}\t{rec:.4f}\t{f1:.4f}\t{spe:.4f}\t{mcc:.4f}\t{auc:.4f}")
print("Confusion [tn fp fn tp]:", tn, fp, fn, tp)

# ----------------------------
# Save adapter + tokenizer
# ----------------------------
adapter_dir = os.path.join(OUT_DIR, "lora_adapter")
os.makedirs(adapter_dir, exist_ok=True)

model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)

print("Saved LoRA adapter + tokenizer to:", adapter_dir)

Train: (1692, 2) {0: 922, 1: 770}
Test : (343, 2) {0: 190, 1: 153}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Step,Training Loss
50,2.268175
100,0.684425
150,0.648428
200,0.626350
250,0.551824
300,0.509433
350,0.397678
400,0.303401
450,0.159849
500,0.103203



KeyboardInterrupt



phi

In [16]:
import os, random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix
)

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ----------------------------
# PATHS
# ----------------------------
TRAIN_PATH = "cvefixes_train_2k_balanced.csv"
TEST_PATH  = "cvefixes_test_400_balanced.csv"

OUT_DIR = "transaction_latest/phi2_CVEvul_qlora"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# CONFIG
# ----------------------------
TEXT_COL  = "code"
LABEL_COL = "label"

MODEL_NAME = "microsoft/phi-2"

SEED = 42
MAX_LEN = 512
EPOCHS = 5
LR = 2e-4
TRAIN_BSZ = 2
GRAD_ACC = 8

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ----------------------------
# Load data
# ----------------------------
train_df = pd.read_csv(TRAIN_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()
test_df  = pd.read_csv(TEST_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()

train_df[LABEL_COL] = pd.to_numeric(train_df[LABEL_COL], errors="coerce")
test_df[LABEL_COL]  = pd.to_numeric(test_df[LABEL_COL], errors="coerce")

train_df = train_df[train_df[LABEL_COL].isin([0, 1])].copy()
test_df  = test_df[test_df[LABEL_COL].isin([0, 1])].copy()

train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)
test_df[LABEL_COL]  = test_df[LABEL_COL].astype(int)

print("Train:", train_df.shape, train_df[LABEL_COL].value_counts().to_dict())
print("Test :", test_df.shape,  test_df[LABEL_COL].value_counts().to_dict())

# ----------------------------
# Prompt
# ----------------------------
def make_prompt(code: str) -> str:
    return (
        "Task: Vulnerability classification.\n"
        "Return a single digit only: 0 (non-vulnerable) or 1 (vulnerable).\n\n"
        "Code:\n"
        f"{code}\n\n"
        "Label:"
    )

# ----------------------------
# Tokenizer
# ----------------------------
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

tok.padding_side = "right"

# single-token label IDs
id_0 = tok.encode("0", add_special_tokens=False)
id_1 = tok.encode("1", add_special_tokens=False)

if len(id_0) != 1 or len(id_1) != 1:
    id_no  = tok.encode(" no", add_special_tokens=False)
    id_yes = tok.encode(" yes", add_special_tokens=False)

    if len(id_no) != 1 or len(id_yes) != 1:
        raise RuntimeError("Neither '0/1' nor ' yes/no' are single-token labels for this tokenizer.")

    print("[WARN] Using ' yes/no' labels instead of 0/1")
    label_token_map = {0: id_no[0], 1: id_yes[0]}

    def make_prompt(code: str) -> str:
        return (
            "Task: Vulnerability classification.\n"
            "Return ONLY one token: ' yes' (vulnerable) or ' no' (non-vulnerable).\n\n"
            "Code:\n"
            f"{code}\n\n"
            "Label:"
        )
else:
    label_token_map = {0: id_0[0], 1: id_1[0]}

# ----------------------------
# Dataset
# ----------------------------
class VulClsCausalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.codes = frame[TEXT_COL].astype(str).tolist()
        self.ys = frame[LABEL_COL].astype(int).tolist()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        code = self.codes[idx]
        y = int(self.ys[idx])

        prompt = make_prompt(code)
        prompt_ids = tok(
            prompt,
            truncation=True,
            max_length=MAX_LEN - 1,
            add_special_tokens=True
        )["input_ids"]

        label_id = int(label_token_map[y])
        input_ids = prompt_ids + [label_id]
        labels = [-100] * len(prompt_ids) + [label_id]
        attn = [1] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attn, labels = [], [], []

    for x in batch:
        pad = max_len - len(x["input_ids"])
        input_ids.append(
            torch.cat([x["input_ids"], torch.full((pad,), tok.pad_token_id, dtype=torch.long)])
        )
        attn.append(
            torch.cat([x["attention_mask"], torch.zeros((pad,), dtype=torch.long)])
        )
        labels.append(
            torch.cat([x["labels"], torch.full((pad,), -100, dtype=torch.long)])
        )

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attn),
        "labels": torch.stack(labels),
    }

train_ds = VulClsCausalDataset(train_df)

# ----------------------------
# Load 4-bit model (QLoRA)
# ----------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
try:
    model.gradient_checkpointing_enable()
except Exception:
    pass

model = prepare_model_for_kbit_training(model)

# ----------------------------
# LoRA for Phi-2
# ----------------------------
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "dense",
        "fc1", "fc2"
    ],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ----------------------------
# Train
# ----------------------------
args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=TRAIN_BSZ,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=50,
    save_steps=300,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collate_fn,
)

trainer.train()

# ----------------------------
# Test inference + metrics
# ----------------------------
@torch.no_grad()
def predict_probs_and_labels(frame: pd.DataFrame, batch_size=8):
    model.eval()
    probs_1, preds = [], []
    y_true = frame[LABEL_COL].astype(int).to_numpy()

    for start in range(0, len(frame), batch_size):
        batch_codes = frame[TEXT_COL].iloc[start:start+batch_size].astype(str).tolist()
        prompts = [make_prompt(c) for c in batch_codes]

        enc = tok(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        ).to(model.device)

        out = model(**enc)

        last_idx = enc["attention_mask"].sum(dim=1) - 1
        logits_last = out.logits[
            torch.arange(out.logits.size(0), device=out.logits.device),
            last_idx,
            :
        ]

        log0 = logits_last[:, label_token_map[0]]
        log1 = logits_last[:, label_token_map[1]]
        p1 = torch.softmax(torch.stack([log0, log1], dim=1), dim=1)[:, 1]

        probs_1.extend(p1.detach().float().cpu().tolist())
        preds.extend((p1 >= 0.5).long().cpu().tolist())

    return np.array(probs_1), np.array(preds), y_true

probs_1, y_pred, y_true = predict_probs_and_labels(test_df, batch_size=8)

acc = accuracy_score(y_true, y_pred)
pre = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1  = f1_score(y_true, y_pred, zero_division=0)
mcc = matthews_corrcoef(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
spe = tn / (tn + fp + 1e-12)

auc = roc_auc_score(y_true, probs_1) if len(np.unique(y_true)) == 2 else float("nan")

print("\nAcc.\tPre.\tRec.\tF1.\tSpe.\tMCC\tAUC")
print(f"{acc:.4f}\t{pre:.4f}\t{rec:.4f}\t{f1:.4f}\t{spe:.4f}\t{mcc:.4f}\t{auc:.4f}")
print("Confusion [tn fp fn tp]:", tn, fp, fn, tp)

# ----------------------------
# Save adapter + tokenizer
# ----------------------------
adapter_dir = os.path.join(OUT_DIR, "lora_adapter")
os.makedirs(adapter_dir, exist_ok=True)

model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)

print("Saved LoRA adapter + tokenizer to:", adapter_dir)

Train: (1692, 2) {0: 922, 1: 770}
Test : (343, 2) {0: 190, 1: 153}


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

trainable params: 23,592,960 || all params: 2,803,276,800 || trainable%: 0.8416


/home/anonymous/.virtualenvs/transaction_latest/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,3.242057
100,0.706422
150,0.686099
200,0.649930
250,0.623176
300,0.536941
350,0.488655
400,0.421730
450,0.313862
500,0.307804


/home/anonymous/.virtualenvs/transaction_latest/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



Acc.	Pre.	Rec.	F1.	Spe.	MCC	AUC
0.7434	0.6946	0.7582	0.7250	0.7316	0.4871	0.7780
Confusion [tn fp fn tp]: 139 51 37 116
Saved LoRA adapter + tokenizer to: transaction_latest/phi2_CVEvul_qlora/lora_adapter


gemma 2b

In [19]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix
)

# ----------------------------
# PATHS
# ----------------------------
TRAIN_PATH = "cvefixes_train_2k_balanced.csv"
TEST_PATH  = "cvefixes_test_400_balanced.csv"

OUT_DIR = "transaction_latest/pytorch_code_classifier"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# CONFIG
# ----------------------------
TEXT_COL = "code"
LABEL_COL = "label"

SEED = 42
MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-3
EMB_DIM = 128
HIDDEN_DIM = 128
MIN_FREQ = 2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ----------------------------
# Load data
# ----------------------------
train_df = pd.read_csv(TRAIN_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()
test_df  = pd.read_csv(TEST_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()

train_df[LABEL_COL] = pd.to_numeric(train_df[LABEL_COL], errors="coerce")
test_df[LABEL_COL]  = pd.to_numeric(test_df[LABEL_COL], errors="coerce")

train_df = train_df[train_df[LABEL_COL].isin([0, 1])].copy()
test_df  = test_df[test_df[LABEL_COL].isin([0, 1])].copy()

train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)
test_df[LABEL_COL]  = test_df[LABEL_COL].astype(int)

print("Train:", train_df.shape, train_df[LABEL_COL].value_counts().to_dict())
print("Test :", test_df.shape, test_df[LABEL_COL].value_counts().to_dict())

# ----------------------------
# Simple tokenizer
# ----------------------------
def code_tokenize(text: str):
    text = str(text)
    # words/operators/punctuation keep together reasonably
    return re.findall(r"[A-Za-z_]\w*|\d+|==|!=|<=|>=|->|&&|\|\||[{}()\[\];,.\+\-/*%<>=!&|^~]", text)

# ----------------------------
# Build vocab
# ----------------------------
counter = Counter()
for code in train_df[TEXT_COL].astype(str).tolist():
    counter.update(code_tokenize(code))

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}
for tok, freq in counter.items():
    if freq >= MIN_FREQ:
        vocab[tok] = len(vocab)

print("Vocab size:", len(vocab))

def encode(text: str, max_len=MAX_LEN):
    toks = code_tokenize(text)
    ids = [vocab.get(t, vocab["<UNK>"]) for t in toks][:max_len]
    attn = [1] * len(ids)

    if len(ids) < max_len:
        pad_len = max_len - len(ids)
        ids += [vocab["<PAD>"]] * pad_len
        attn += [0] * pad_len

    return ids, attn

# ----------------------------
# Dataset
# ----------------------------
class CodeDataset(Dataset):
    def __init__(self, df):
        self.codes = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].astype(int).tolist()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        ids, attn = encode(self.codes[idx], MAX_LEN)
        y = self.labels[idx]
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.float32),
            "labels": torch.tensor(y, dtype=torch.float32),
        }

train_ds = CodeDataset(train_df)
test_ds = CodeDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# ----------------------------
# Model: BiLSTM classifier
# ----------------------------
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)                     # [B, L, E]
        out, _ = self.lstm(x)                             # [B, L, 2H]

        mask = attention_mask.unsqueeze(-1)               # [B, L, 1]
        out = out * mask

        summed = out.sum(dim=1)                           # [B, 2H]
        lengths = mask.sum(dim=1).clamp(min=1e-6)         # [B, 1]
        pooled = summed / lengths                         # masked mean pooling

        pooled = self.dropout(pooled)
        logits = self.fc(pooled).squeeze(-1)              # [B]
        return logits

model = BiLSTMClassifier(
    vocab_size=len(vocab),
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# ----------------------------
# Train
# ----------------------------
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / max(len(train_loader), 1)
    print(f"Epoch {epoch}/{EPOCHS} - Train Loss: {avg_loss:.4f}")

# ----------------------------
# Evaluate
# ----------------------------
model.eval()
all_probs = []
all_preds = []
all_true = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].cpu().numpy()

        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_true.extend(labels.astype(int).tolist())

y_true = np.array(all_true)
y_pred = np.array(all_preds)
probs_1 = np.array(all_probs)

acc = accuracy_score(y_true, y_pred)
pre = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1  = f1_score(y_true, y_pred, zero_division=0)
mcc = matthews_corrcoef(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
spe = tn / (tn + fp + 1e-12)
auc = roc_auc_score(y_true, probs_1) if len(np.unique(y_true)) == 2 else float("nan")

print("\nAcc.\tPre.\tRec.\tF1.\tSpe.\tMCC\tAUC")
print(f"{acc:.4f}\t{pre:.4f}\t{rec:.4f}\t{f1:.4f}\t{spe:.4f}\t{mcc:.4f}\t{auc:.4f}")
print("Confusion [tn fp fn tp]:", tn, fp, fn, tp)

# ----------------------------
# Save
# ----------------------------
save_path = os.path.join(OUT_DIR, "bilstm_code_classifier.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "vocab": vocab,
    "config": {
        "vocab_size": len(vocab),
        "emb_dim": EMB_DIM,
        "hidden_dim": HIDDEN_DIM,
        "max_len": MAX_LEN
    }
}, save_path)

print("Saved model to:", save_path)

Train: (1692, 2) {0: 922, 1: 770}
Test : (343, 2) {0: 190, 1: 153}
Vocab size: 80702
Epoch 1/10 - Train Loss: 0.6837
Epoch 2/10 - Train Loss: 0.6461
Epoch 3/10 - Train Loss: 0.5826
Epoch 4/10 - Train Loss: 0.5015
Epoch 5/10 - Train Loss: 0.3944
Epoch 6/10 - Train Loss: 0.3121
Epoch 7/10 - Train Loss: 0.2554
Epoch 8/10 - Train Loss: 0.1879
Epoch 9/10 - Train Loss: 0.1289
Epoch 10/10 - Train Loss: 0.0835

Acc.	Pre.	Rec.	F1.	Spe.	MCC	AUC
0.6152	0.5714	0.5490	0.5600	0.6684	0.2184	0.6363
Confusion [tn fp fn tp]: 127 63 69 84
Saved model to: transaction_latest/pytorch_code_classifier/bilstm_code_classifier.pt


In [4]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix
)

from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer
)

# ----------------------------
# PATHS
# ----------------------------
TRAIN_PATH = "cvefixes_train_2k_balanced.csv"
TEST_PATH  = "cvefixes_test_400_balanced.csv"

OUT_DIR = "transaction_latest/unixcoder_CVEvul_cls"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# CONFIG
# ----------------------------
TEXT_COL  = "code"
LABEL_COL = "label"

MODEL_NAME = "microsoft/unixcoder-base"

SEED = 42
MAX_LEN = 512
EPOCHS = 5
LR = 2e-5
TRAIN_BSZ = 8
EVAL_BSZ = 8

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ----------------------------
# Load data
# ----------------------------
train_df = pd.read_csv(TRAIN_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()
test_df  = pd.read_csv(TEST_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()

train_df[LABEL_COL] = pd.to_numeric(train_df[LABEL_COL], errors="coerce")
test_df[LABEL_COL]  = pd.to_numeric(test_df[LABEL_COL], errors="coerce")

train_df = train_df[train_df[LABEL_COL].isin([0, 1])].copy()
test_df  = test_df[test_df[LABEL_COL].isin([0, 1])].copy()

train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)
test_df[LABEL_COL]  = test_df[LABEL_COL].astype(int)

print("Train:", train_df.shape, train_df[LABEL_COL].value_counts().to_dict())
print("Test :", test_df.shape,  test_df[LABEL_COL].value_counts().to_dict())

# ----------------------------
# Tokenizer
# ----------------------------
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------
# Dataset
# ----------------------------
class VulDataset(Dataset):
    def __init__(self, frame):
        self.codes = frame[TEXT_COL].astype(str).tolist()
        self.labels = frame[LABEL_COL].astype(int).tolist()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        code = self.codes[idx]
        y = self.labels[idx]

        enc = tok(
            code,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(y, dtype=torch.long)
        }
        return item

train_ds = VulDataset(train_df)
test_ds  = VulDataset(test_df)

# ----------------------------
# Model
# ----------------------------
class UniXcoderClassifier(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # CLS token representation
        pooled = outputs.last_hidden_state[:, 0, :]   # [B, H]
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

model = UniXcoderClassifier(MODEL_NAME)

# ----------------------------
# Metrics
# ----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    pre = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1  = f1_score(labels, preds, zero_division=0)
    mcc = matthews_corrcoef(labels, preds)

    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    spe = tn / (tn + fp + 1e-12)

    auc = roc_auc_score(labels, probs) if len(np.unique(labels)) == 2 else float("nan")

    return {
        "accuracy": acc,
        "precision": pre,
        "recall": rec,
        "f1": f1,
        "specificity": spe,
        "mcc": mcc,
        "auc": auc
    }

# ----------------------------
# Train
# ----------------------------
args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=TRAIN_BSZ,
    per_device_eval_batch_size=EVAL_BSZ,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

trainer.train()

# ----------------------------
# Final evaluation
# ----------------------------
pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
y_true = pred_out.label_ids

probs_1 = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
y_pred = np.argmax(logits, axis=1)

acc = accuracy_score(y_true, y_pred)
pre = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1  = f1_score(y_true, y_pred, zero_division=0)
mcc = matthews_corrcoef(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
spe = tn / (tn + fp + 1e-12)
auc = roc_auc_score(y_true, probs_1) if len(np.unique(y_true)) == 2 else float("nan")

print("\nAcc.\tPre.\tRec.\tF1.\tSpe.\tMCC\tAUC")
print(f"{acc:.4f}\t{pre:.4f}\t{rec:.4f}\t{f1:.4f}\t{spe:.4f}\t{mcc:.4f}\t{auc:.4f}")
print("Confusion [tn fp fn tp]:", tn, fp, fn, tp)

# ----------------------------
# Save
# ----------------------------
save_dir = os.path.join(OUT_DIR, "final_model")
os.makedirs(save_dir, exist_ok=True)

trainer.save_model(save_dir)
tok.save_pretrained(save_dir)

print("Saved model + tokenizer to:", save_dir)

Train: (1692, 2) {0: 922, 1: 770}
Test : (343, 2) {0: 190, 1: 153}


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/unixcoder-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Specificity,Mcc,Auc
1,0.650653,0.702336,0.603499,0.537445,0.797386,0.642105,0.447368,0.257164,0.682370
2,0.531545,0.651778,0.658892,0.601124,0.699346,0.646526,0.626316,0.323995,0.732559
3,0.349598,0.949160,0.679300,0.631902,0.673203,0.651899,0.684211,0.355765,0.752408
4,0.148969,1.425455,0.667638,0.601036,0.758170,0.670520,0.594737,0.353637,0.747007
5,0.073339,1.546249,0.664723,0.623377,0.627451,0.625407,0.694737,0.321989,0.747730



Acc.	Pre.	Rec.	F1.	Spe.	MCC	AUC
0.6676	0.6010	0.7582	0.6705	0.5947	0.3536	0.7470
Confusion [tn fp fn tp]: 113 77 37 116
Saved model + tokenizer to: transaction_latest/unixcoder_CVEvul_cls/final_model


gb

In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix
)

from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer
)

# ----------------------------
# PATHS
# ----------------------------
TRAIN_PATH = "cvefixes_merged_verified_only_train.csv"
TEST_PATH  = "cvefixes_merged_verified_only_test.csv"

OUT_DIR = "transaction_latest/graphcodebert_vCVEvul_cls"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# CONFIG
# ----------------------------
TEXT_COL  = "code"
LABEL_COL = "label"

MODEL_NAME = "microsoft/graphcodebert-base"

SEED = 42
MAX_LEN = 512
EPOCHS = 5
LR = 2e-5
TRAIN_BSZ = 8
EVAL_BSZ = 8

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ----------------------------
# Load data
# ----------------------------
train_df = pd.read_csv(TRAIN_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()
test_df  = pd.read_csv(TEST_PATH)[[TEXT_COL, LABEL_COL]].dropna().copy()

train_df[LABEL_COL] = pd.to_numeric(train_df[LABEL_COL], errors="coerce")
test_df[LABEL_COL]  = pd.to_numeric(test_df[LABEL_COL], errors="coerce")

train_df = train_df[train_df[LABEL_COL].isin([0, 1])].copy()
test_df  = test_df[test_df[LABEL_COL].isin([0, 1])].copy()

train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)
test_df[LABEL_COL]  = test_df[LABEL_COL].astype(int)

print("Train:", train_df.shape, train_df[LABEL_COL].value_counts().to_dict())
print("Test :", test_df.shape,  test_df[LABEL_COL].value_counts().to_dict())

# ----------------------------
# Tokenizer
# ----------------------------
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------
# Dataset
# ----------------------------
class VulDataset(Dataset):
    def __init__(self, frame):
        self.codes = frame[TEXT_COL].astype(str).tolist()
        self.labels = frame[LABEL_COL].astype(int).tolist()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        code = self.codes[idx]
        y = self.labels[idx]

        enc = tok(
            code,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(y, dtype=torch.long)
        }

train_ds = VulDataset(train_df)
test_ds  = VulDataset(test_df)

# ----------------------------
# Model
# ----------------------------
class GraphCodeBERTClassifier(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # CLS token representation
        pooled = outputs.last_hidden_state[:, 0, :]   # [B, H]
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

model = GraphCodeBERTClassifier(MODEL_NAME)

# ----------------------------
# Metrics
# ----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    pre = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1  = f1_score(labels, preds, zero_division=0)
    mcc = matthews_corrcoef(labels, preds)

    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    spe = tn / (tn + fp + 1e-12)

    auc = roc_auc_score(labels, probs) if len(np.unique(labels)) == 2 else float("nan")

    return {
        "accuracy": acc,
        "precision": pre,
        "recall": rec,
        "f1": f1,
        "specificity": spe,
        "mcc": mcc,
        "auc": auc
    }

# ----------------------------
# Train
# ----------------------------
args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=TRAIN_BSZ,
    per_device_eval_batch_size=EVAL_BSZ,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

trainer.train()

# ----------------------------
# Final evaluation
# ----------------------------
pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
y_true = pred_out.label_ids

probs_1 = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
y_pred = np.argmax(logits, axis=1)

acc = accuracy_score(y_true, y_pred)
pre = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1  = f1_score(y_true, y_pred, zero_division=0)
mcc = matthews_corrcoef(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
spe = tn / (tn + fp + 1e-12)
auc = roc_auc_score(y_true, probs_1) if len(np.unique(y_true)) == 2 else float("nan")

print("\nAcc.\tPre.\tRec.\tF1.\tSpe.\tMCC\tAUC")
print(f"{acc:.4f}\t{pre:.4f}\t{rec:.4f}\t{f1:.4f}\t{spe:.4f}\t{mcc:.4f}\t{auc:.4f}")
print("Confusion [tn fp fn tp]:", tn, fp, fn, tp)

# ----------------------------
# Save
# ----------------------------
save_dir = os.path.join(OUT_DIR, "final_model")
os.makedirs(save_dir, exist_ok=True)

trainer.save_model(save_dir)
tok.save_pretrained(save_dir)

print("Saved model + tokenizer to:", save_dir)

Train: (4329, 2) {0: 2275, 1: 2054}
Test : (1091, 2) {0: 603, 1: 488}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
